<a href="https://colab.research.google.com/github/itsdragon3990-dev/datajam-2026-airbnb/blob/main/Datajam_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_csv("listings.csv")
print("Starting listings:", len(df))

Starting listings: 6509


In [ ]:
df["price"] = df["price"].str.replace(r"[$,]", "", regex=True).astype(float)
df["last_review"] = pd.to_datetime(df["last_review"], format="%Y-%m-%d", errors="coerce")
print(df["price"].describe())

count     5845.000000
mean       405.609743
std       1269.704113
min          6.170000
25%        156.000000
50%        318.000000
75%        496.730000
max      91294.330000
Name: price, dtype: float64


In [ ]:
df = df[df["room_type"] == "Entire home/apt"];  print("Entire homes:", len(df))
df = df[df["price"].notna()];                   print("Has a price:", len(df))
df = df[df["number_of_reviews_ltm"] > 0];       print("Booked in last year:", len(df))
df = df[df["minimum_nights"] < 30];             print("Short-term only:", len(df))

Entire homes: 5322
Has a price: 4796
Booked in last year: 3604
Short-term only: 3230


In [ ]:
cap = df["price"].quantile(0.99)
print("Price cap:", round(cap))
df = df[df["price"] <= cap]
print("Final clean listings:", len(df))

Price cap: 1994
Final clean listings: 3197


In [ ]:
raw = pd.read_csv("listings.csv")
raw["price"] = raw["price"].str.replace(r"[$,]", "", regex=True).astype(float)
no_activity_filter = raw[(raw["room_type"] == "Entire home/apt") &
                         (raw["price"].notna()) &
                         (raw["minimum_nights"] < 30) &
                         (raw["price"] <= cap)]
print("WITHOUT activity filter, median revenue:", no_activity_filter["estimated_revenue_l365d"].median())
print("WITH activity filter, median revenue:   ", df["estimated_revenue_l365d"].median())

WITHOUT activity filter, median revenue: 32991.5
WITH activity filter, median revenue:    42177.0


In [ ]:
df.to_csv("listings_clean.csv", index=False)
print("Saved!")

Saved!


In [ ]:
import numpy as np
condo = ["Entire condo", "Entire rental unit", "Entire loft", "Entire serviced apartment"]
house = ["Entire home", "Entire townhouse", "Entire villa", "Entire bungalow"]
df["type"] = np.select([df["property_type"].isin(condo), df["property_type"].isin(house)],
                       ["condo", "house"], "other")
df = df[df["type"] != "other"]
print(df["type"].value_counts())
print(df.groupby("type")["estimated_revenue_l365d"].quantile([0.25, 0.5, 0.75]).unstack())

type
condo    1344
house    1190
Name: count, dtype: int64
           0.25     0.50     0.75
type                             
condo  17935.75  48289.5  84390.0
house  12825.75  35519.0  68681.0


In [ ]:
areas = (df[df["type"] == "condo"].groupby("neighbourhood_cleansed")["estimated_revenue_l365d"]
           .agg(["count", "median"]).query("count >= 20")
           .sort_values("median", ascending=False))
print(areas)

                        count   median
neighbourhood_cleansed                
Downtown                  768  59854.5
Downtown Eastside         249  45390.0
Kitsilano                  51  41247.0
West End                   73  40608.0
Mount Pleasant             72  29886.0
Grandview-Woodland         24  17403.5


In [ ]:
tax = pd.read_csv("property-tax-report.csv", sep=";", low_memory=False)
tax.columns = tax.columns.str.lower()
tax["value"] = tax["current_land_value"] + tax["current_improvement_value"]

condos_tax = tax[(tax["legal_type"] == "STRATA") & (tax["value"] > 200000)]
houses_tax = tax[(tax["legal_type"] == "LAND") & tax["zoning_district"].str.startswith(("R1", "RS"), na=False)]

print("Typical condo value:", condos_tax["value"].median(), "| tax:", condos_tax["tax_levy"].median())
print("Typical house value:", houses_tax["value"].median(), "| tax:", houses_tax["tax_levy"].median())

downtown = condos_tax[condos_tax["property_postal_code"].str[:3].isin(["V6B","V6C","V6E","V6G","V6Z"])]
print("Downtown condo value:", downtown["value"].median())

Typical condo value: 801000.0 | tax: 2778.62
Typical house value: 2412000.0 | tax: 10216.6
Downtown condo value: 761000.0


In [ ]:
price = {"condo": condos_tax["value"].median(), "house": houses_tax["value"].median()}
prop_tax = {"condo": condos_tax["tax_levy"].median(), "house": houses_tax["tax_levy"].median()}
LICENCE = 1108 + 450    # city licence + provincial registration
MGMT = 0.30             # 10% management + cleaning (~$120 per stay)
OTHER = {"condo": 7000, "house": 9000}   # strata fees, insurance, repairs (assumption)

for t in ["condo", "house"]:
    rev = df[df["type"] == t]["estimated_revenue_l365d"].median()
    cost = LICENCE + MGMT * rev + OTHER[t] + prop_tax[t]
    buy = price[t] * 1.10   # assessed value is usually below market, so add 10%
    print(t, "| revenue:", round(rev), "| profit:", round(rev - cost),
          "| buy price:", round(buy), "| net yield:", round((rev - cost) / buy * 100, 2), "%")

condo | revenue: 48290 | profit: 22466 | buy price: 881100 | net yield: 2.55 %
house | revenue: 35519 | profit: 4089 | buy price: 2653200 | net yield: 0.15 %


In [ ]:
for t in ["condo", "house"]:
    buy = price[t] * 1.10
    units = int(10_000_000 // buy)
    r = df[df["type"] == t]["estimated_revenue_l365d"]
    for name, q in [("low", 0.25), ("expected", 0.5), ("high", 0.75)]:
        rev = r.quantile(q)
        profit = rev - (LICENCE + MGMT * rev + OTHER[t] + prop_tax[t])
        print(t, units, "units |", name, "yearly profit:", round(units * profit))

condo 11 units | low yearly profit: 13402
condo 11 units | expected yearly profit: 247126
condo 11 units | high yearly profit: 525100
house 3 units | low yearly profit: -35390
house 3 units | expected yearly profit: 12266
house 3 units | high yearly profit: 81906


In [ ]:
r = df[df["type"] == "condo"]["estimated_revenue_l365d"].median() * 0.8
units = int(10_000_000 // (price["condo"] * 1.10))
print("If bookings drop 20%:", round(units * (r - (LICENCE + MGMT * r + OTHER["condo"] + prop_tax["condo"]))))

If bookings drop 20%: 172760
